<a href="https://colab.research.google.com/github/hanauert/Hausarbeit-GenAI/blob/main/03_stance_detection_NLI_classifiers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Load gold annotated dataset**

In [27]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import matthews_corrcoef

In [28]:
from google.colab import drive

# Mount your Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
subsample_gold = pd.read_csv('/content/drive/MyDrive/Hausarbeit_GenAI_Hanauer/data/gold_annotated.csv')

In [30]:
adj_to_base = {
    1:'positiv',
    2:'negativ',
    3:'neutral',
}

# Apply mapping
subsample_gold['gold_standard'] = subsample_gold['gold_standard'].map(adj_to_base)

In [31]:
subsample_gold['lead'] = subsample_gold['body_text'].str.split('\n\n|\n').str[0]

In [32]:
subsample_gold.head()

,metadata,body_text,title,date,month,newspaper,article_id,paragraph,gold_standard,lead
0,Wir verschenken zu viel Zeit\n \nFrankfurter R...,Hochwasser in Niedersachsen im Januar 2024. dp...,Wir verschenken zu viel Zeit,2025-01-02,2025-01,FR,A_1870,Das gilt übrigens auch für die demokratischen ...,positiv,Hochwasser in Niedersachsen im Januar 2024. dpa
1,Das Armutsrisiko in Deutschland ist rückläufig...,Im Wahlkampf nimmt die Debatte um Arm und Reic...,Das Armutsrisiko in Deutschland ist rückläufig...,2024-12-26,2024-12,Welt,A_291,Trotz des gewachsenen Migrantenanteils in der ...,neutral,Im Wahlkampf nimmt die Debatte um Arm und Reic...
2,Arbeitgeber erwarten Wohlstandsverlust durch P...,Bahnverbindungen und Unterrichtsstunden fallen...,Arbeitgeber erwarten Wohlstandsverlust durch P...,2023-11-26,2023-11,Welt,B_22,Die Unionsfraktion hatte stattdessen vorgeschl...,negativ,Bahnverbindungen und Unterrichtsstunden fallen...
3,Das Ende einer Erfolgsstory; Kanzler Scholz fe...,"Es ist einer der Textbausteine, die Bundeskanz...",Das Ende einer Erfolgsstory; Kanzler Scholz fe...,2024-11-25,2024-11,Welt,A_303,Hauptgründe für die Aufwärtsbewegung sind die ...,positiv,"Es ist einer der Textbausteine, die Bundeskanz..."
4,„Investitionen entscheiden über die Leistungsf...,Die neue Landesregierung ist mit dem Versprech...,„Investitionen entscheiden über die Leistungsf...,2024-09-10,2024-09,FR,B_408,Die Aufnahme einer Erwerbsarbeit ist die beste...,positiv,Die neue Landesregierung ist mit dem Versprech...


##**Define Metrics**

In [33]:
from sklearn.metrics import cohen_kappa_score, f1_score, accuracy_score, precision_score, recall_score

def evaluate_model(true, pred):
    return {
        'Cohens Kappa': cohen_kappa_score(true, pred),
        'F1': f1_score(true, pred, average='weighted'),
        'Accuracy': accuracy_score(true, pred),
        'Precision': precision_score(true, pred, average='weighted'),
        'Recall': recall_score(true, pred, average='weighted')
    }

#**Evaluate NLI Classifiers**


#**MoritzLaurer/bge-m3-zeroshot-v2.0**

In [34]:
classifier = pipeline("zero-shot-classification", model='MoritzLaurer/bge-m3-zeroshot-v2.0')



Device set to use cuda:0


###**Annotate Title + Paragraph**

In [35]:
	samples = list("Title:\n" + subsample_gold['title'] + "\n\n" + "Paragraph:\n" + subsample_gold['paragraph'])
	template = "Der Text impliziert eine {} Meinung zu Migration in Bezug auf Fachkräftemangel."
	labels = ['positive', 'negative', 'keine']

In [36]:
# classify the documents
res = classifier(samples, labels, hypothesis_template = template, multi_label = False)

# return the most probable label and add it to our the frame
subsample_gold['label_ml_p6'] = [label['labels'][0] for label in res]

subsample_gold['score_ml_p6'] = [r['scores'][0] for r in res]

In [37]:
adj_to_base = {
    'positive': 'positiv',
    'negative': 'negativ',
    'keine': 'neutral',
}

# Apply mapping
subsample_gold['label_ml_p6'] = subsample_gold['label_ml_p6'].map(adj_to_base)

In [38]:
metrics_ml = evaluate_model(subsample_gold['gold_standard'], subsample_gold['label_ml_p6'])

print("Evaluation Metrics: MoritzLaurer/bge-m3-zeroshot-v2.0")
for metric, value in metrics_ml.items():
    print(f"{metric}: {value:.3f}")

Evaluation Metrics: MoritzLaurer/bge-m3-zeroshot-v2.0
Cohens Kappa: 0.228
F1: 0.452
Accuracy: 0.460
Precision: 0.564
Recall: 0.460


#**mlburnham/Political_DEBATE_DeBERTa_large_v1.1**

In [39]:
classifier_mlb = pipeline("zero-shot-classification", model='mlburnham/Political_DEBATE_DeBERTa_large_v1.1')

Device set to use cuda:0


###**Annotate Title + Paragraph**

In [40]:
samples = list("Title:\n" + subsample_gold['title'] + "\n\n" + subsample_gold['paragraph'])
template = 'Der Text positioniert sich {} zu Migration.'
labels = ['positiv', 'negativ', 'neutral']

In [41]:
# classify the documents
res = classifier_mlb(samples, labels, hypothesis_template = template, multi_label = False)

# return the most probable label and add it to the data frame
subsample_gold['label_mlb_p8'] = [label['labels'][0] for label in res]

subsample_gold['score_mlb_p8'] = [r['scores'][0] for r in res]

In [42]:
metrics_mlb = evaluate_model(subsample_gold['gold_standard'], subsample_gold['label_mlb_p8'])

print("Evaluation Metrics: mlburnham/Political_DEBATE_DeBERTa_large_v1.1")
for metric, value in metrics_mlb.items():
    print(f"{metric}: {value:.3f}")

Evaluation Metrics: mlburnham/Political_DEBATE_DeBERTa_large_v1.1
Cohens Kappa: 0.379
F1: 0.578
Accuracy: 0.584
Precision: 0.621
Recall: 0.584


In [43]:
from sklearn.metrics import classification_report
print(classification_report(subsample_gold['gold_standard'], subsample_gold['label_mlb_p8']))

              precision    recall  f1-score   support

     negativ       0.67      0.66      0.67        62
     neutral       0.69      0.41      0.52       109
     positiv       0.48      0.76      0.59        79

    accuracy                           0.58       250
   macro avg       0.62      0.61      0.59       250
weighted avg       0.62      0.58      0.58       250



#**luissattelmayer/NLI-stance-immigration-burnham-finetuned-german**

In [44]:
classifier_ls = pipeline ('zero-shot-classification', model='luissattelmayer/NLI-stance-immigration-burnham-finetuned-german')


Device set to use cuda:0


###**Annotate Title + Paragraph**

In [45]:
samples = list("Title:\n" + subsample_gold['title'] + "\n\n" + "Paragraph:\n" + subsample_gold['paragraph'])
template = "Der Text impliziert eine {} Meinung zu Migration in Bezug auf Fachkräftemangel."
labels = ['positive', 'negative', 'keine']

In [46]:
# classify the documents
res = classifier_ls(samples, labels, hypothesis_template = template, multi_label = False)

# return the most probable label and add it to the data frame
subsample_gold['label_ls_p6'] = [label['labels'][0] for label in res]

subsample_gold['score_ls_p6'] = [r['scores'][0] for r in res]

In [47]:
adj_to_base = {
    'positive': 'positiv',
    'negative': 'negativ',
    'keine': 'neutral',
}

# Apply mapping
subsample_gold['label_ls_p6'] = subsample_gold['label_ls_p6'].map(adj_to_base)

In [48]:
metrics_ls = evaluate_model(subsample_gold['gold_standard'], subsample_gold['label_ls_p6'])

print("Evaluation Metrics: luissattelmayer/NLI-stance-immigration-burnham-finetuned-german")
for metric, value in metrics_ls.items():
    print(f"{metric}: {value:.3f}")

Evaluation Metrics: luissattelmayer/NLI-stance-immigration-burnham-finetuned-german
Cohens Kappa: 0.195
F1: 0.389
Accuracy: 0.428
Precision: 0.538
Recall: 0.428


#**Evaluation**

In [49]:
metrics_df = pd.DataFrame([metrics_ml, metrics_mlb, metrics_ls],
                          index=['MoritzLaurer', 'mlburnham', 'luissattelmayer'])

print(metrics_df.round(3))

                 Cohens Kappa     F1  Accuracy  Precision  Recall
MoritzLaurer            0.228  0.452     0.460      0.564   0.460
mlburnham               0.379  0.578     0.584      0.621   0.584
luissattelmayer         0.195  0.389     0.428      0.538   0.428


###**Save Results**

In [51]:
subsample_gold[['date', 'title', 'paragraph', 'newspaper', 'gold_standard', 'label_ml_p6', 'score_ml_p6', 'label_mlb_p8','score_mlb_p8', 'label_ls_p6', 'score_ls_p6' ]].to_csv('/content/drive/MyDrive/Hausarbeit_GenAI_Hanauer/data/gold_annotated_vs_LLM.csv', index=False)